# 02 - Excess neonatal mortality beyond structure (Paper 1)

For each municipality we model the neonatal deaths **expected from its structure**
(socioeconomic profile, then + access) and read the **residual** - who dies more than
structurally-similar peers. Extends Martinelli et al. (2025) from one metropolis / adults to
**all Brazilian municipalities / neonatal**, replacing their age-based expectation with a
socioeconomic + access one (there is no age confounder per live birth).

**Design.** Negative-binomial GLM, outcome = deaths, offset = log(live births); nested
expectation A (socioeconomic) then B (+ access); SMR = observed / expected with Byar 95% CI and
empirical-Bayes shrinkage; **significant excess = CI lower bound > 1** (p<0.05). A cause layer
locates the excess across action groups (avoidability); the avoidable burden is reported as a
**ladder over one base**, from the structural mean to the best structural peers. Framing is
**place-level, cause-specific - never an individual counterfactual.**

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os
import numpy as np, pandas as pd, csv
from collections import defaultdict
import statsmodels.api as sm, statsmodels.formula.api as smf

# resolve repo root relative to this notebook (works across the two synced copies)
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC = f'{ROOT}/data/processed'; RAW = f'{ROOT}/data/raw'
ACTION = ['prenatal', 'delivery', 'newborn', 'malformation']
Z = 1.959963985  # 95%

## 1. Assemble the municipality table (pooled 2014-2024)

In [2]:
base = pd.read_csv(f'{PROC}/muni_base_2014_2024.csv', dtype={'CODMUNRES': str})
idhm = pd.read_csv(f'{PROC}/idhm_muni.csv', dtype={'CODMUNRES': str})
pop = (pd.read_csv(f'{PROC}/populacao_muni_ano.csv', dtype={'CODMUNRES': str})
         .groupby('CODMUNRES').populacao.mean().rename('pop_mean'))
cn = pd.read_csv(f'{PROC}/cnes_muni_ano.csv', dtype={'CODMUNRES': str})
cn['CODMUNRES'] = cn['CODMUNRES'].str[:6]
acc = cn.groupby('CODMUNRES').agg(ubs=('ubs', 'mean'), nicu_beds=('uti_neo_leitos', 'mean'))
last = cn.sort_values('ANO').groupby('CODMUNRES').last()

# distance to nearest NICU-holding municipality (haversine between CNES centroids)
la = defaultdict(list); lo = defaultdict(list)
with open(f'{RAW}/cnes_estabelecimentos.csv', encoding='latin-1') as fh:
    r = csv.reader(fh, delimiter=';')
    h = [x.strip().strip('"').upper() for x in next(r)]; ix = {x: i for i, x in enumerate(h)}
    for row in r:
        if len(row) <= ix['NU_LONGITUDE']:
            continue
        try:
            a = float(row[ix['NU_LATITUDE']]); o = float(row[ix['NU_LONGITUDE']])
        except ValueError:
            continue
        if -34 <= a <= 6 and -74 <= o <= -34:
            ib = row[ix['CO_IBGE']].strip().strip('"')[:6]; la[ib].append(a); lo[ib].append(o)
cent = {ib: (np.median(la[ib]), np.median(lo[ib])) for ib in la}
nc = np.array([cent[m] for m in set(last.index[last['uti_neo_leitos'] > 0]) if m in cent])

def hav(a1, o1, A, O):
    R = 6371; p = np.pi / 180; d1 = (A - a1) * p; d2 = (O - o1) * p
    return 2 * R * np.arcsin(np.sqrt(np.sin(d1 / 2) ** 2 + np.cos(a1 * p) * np.cos(A * p) * np.sin(d2 / 2) ** 2))

def dist(m):
    if m not in cent:
        return np.nan
    a, o = cent[m]; return float(hav(a, o, nc[:, 0], nc[:, 1]).min())

df = (base.merge(idhm[['CODMUNRES', 'municipio', 'uf', 'idhm', 'idhm_renda', 'idhm_long', 'idhm_educ']],
                 on='CODMUNRES', how='left')
          .merge(pop, on='CODMUNRES', how='left')
          .merge(acc, on='CODMUNRES', how='left'))
df['dist_nicu_km'] = df['CODMUNRES'].map(dist)
df['REG'] = df['CODMUNRES'].str[0].map({'1': 'North', '2': 'Northeast', '3': 'Southeast',
                                        '4': 'South', '5': 'Central-West'})
df = df[(df.live_births > 0) & df.idhm_renda.notna() & df.pop_mean.notna()].copy()
df['logbirths'] = np.log(df.live_births)
df['logpop'] = np.log(df.pop_mean)
df['nicu_per1k'] = df.nicu_beds.fillna(0) / df.live_births * 1000
df['ubs_per1k'] = df.ubs.fillna(0) / df.live_births * 1000
df['dist_nicu_km'] = df.dist_nicu_km.fillna(df.dist_nicu_km.median())
df['illdef_share'] = df.illdef / (df.deaths_total + df.illdef).replace(0, np.nan)
print(f'modeling sample: {len(df)} municipalities, {int(df.deaths_total.sum()):,} neonatal deaths')

modeling sample: 5477 municipalities, 259,965 neonatal deaths


## 1b. Descriptive epidemiology (full cohort)

National and regional neonatal mortality, cause composition, and the death-count reliability tiers that
motivate the >= 50 threshold, on the full cohort (before the modelling restriction).

In [3]:
# full cohort (5,481 municipalities): national / regional NMR, cause mix, reliability tiers
base['REG'] = base['CODMUNRES'].str[0].map({'1': 'North', '2': 'Northeast', '3': 'Southeast',
                                            '4': 'South', '5': 'Central-West'})
tot_d, tot_b = base.deaths_total.sum(), base.live_births.sum()
print(f'national: {int(tot_d):,} neonatal deaths / {int(tot_b):,} live births | NMR {tot_d / tot_b * 1000:.2f} per 1,000')
print('NMR by region (deaths, rate):')
gg = base.groupby('REG').agg(deaths=('deaths_total', 'sum'), births=('live_births', 'sum'))
gg['nmr'] = gg.deaths / gg.births * 1000
for r in ['North', 'Northeast', 'Central-West', 'Southeast', 'South']:
    print(f'  {r:13s}: {int(gg.loc[r, "deaths"]):>7,}  NMR {gg.loc[r, "nmr"]:.2f}')
print('cause composition (% of deaths):',
      {c: round(base[c].sum() / tot_d * 100, 1) for c in ACTION + ['illdef']})
print('reliability tiers:')
for lab, lo, hi in [('high (>=50)', 50, 1e9), ('medium (20-49)', 20, 50), ('low (<20)', 0, 20)]:
    m = (base.deaths_total >= lo) & (base.deaths_total < hi)
    print(f'  {lab:15s}: {int(m.sum()):>5,} munis ({base.loc[m, "deaths_total"].sum() / tot_d * 100:.1f}% of deaths)')

national: 260,023 neonatal deaths / 30,437,646 live births | NMR 8.54 per 1,000
NMR by region (deaths, rate):
  North        :  33,595  NMR 10.05
  Northeast    :  82,490  NMR 9.65
  Central-West :  21,562  NMR 8.36
  Southeast    :  92,634  NMR 7.84
  South        :  29,742  NMR 7.16
cause composition (% of deaths): {'prenatal': np.float64(28.3), 'delivery': np.float64(7.7), 'newborn': np.float64(37.5), 'malformation': np.float64(21.0), 'illdef': np.float64(5.5)}
reliability tiers:
  high (>=50)    :   955 munis (74.9% of deaths)
  medium (20-49) : 1,250 munis (15.0% of deaths)
  low (<20)      : 3,276 munis (10.2% of deaths)


## 2. Fit helper

Negative-binomial GLM; the dispersion `alpha` is estimated by method of moments from a Poisson
pass (auxiliary OLS of the squared-residual statistic on the fitted mean).

In [4]:
def fit_nb(formula, d, off, outcome='deaths_total'):
    poi = smf.glm(formula, data=d, family=sm.families.Poisson(), offset=off).fit()
    mu = poi.mu
    aux = ((d[outcome] - mu) ** 2 - d[outcome]) / mu
    alpha = max(sm.OLS(aux, mu).fit().params[0], 1e-6)
    nb = smf.glm(formula, data=d, family=sm.families.NegativeBinomial(alpha=alpha), offset=off).fit()
    disp = poi.pearson_chi2 / poi.df_resid
    return nb, alpha, disp

## 3. Model A - socioeconomic structure

In [5]:
fA = 'deaths_total ~ idhm_renda + idhm_educ + logpop'
nbA, alphaA, disp = fit_nb(fA, df, df.logbirths.values)
df['expA'] = nbA.mu; df['SMR_A'] = df.deaths_total / df.expA
print(f'Poisson dispersion = {disp:.1f}  -> negative binomial justified (alpha={alphaA:.3f})')
print('MODEL A (socioeconomic):', nbA.params.round(3).to_dict())
print(f'  pseudo-R2 (deviance) = {1 - nbA.deviance / nbA.null_deviance:.3f}')

Poisson dispersion = 2.3  -> negative binomial justified (alpha=0.012)
MODEL A (socioeconomic): {'Intercept': -4.067, 'idhm_renda': -0.972, 'idhm_educ': -0.457, 'logpop': 0.022}
  pseudo-R2 (deviance) = 0.162


## 4. Model B - + access (nested)

Does the excess survive adjustment for infrastructure? If SMR is barely moved by adding distance
to NICU, NICU beds and primary-care units, the residual is not "they are far from a NICU" - it is
a genuine quality/effectiveness gap.

In [6]:
fB = fA + ' + dist_nicu_km + nicu_per1k + ubs_per1k'
nbB, alphaB, _ = fit_nb(fB, df, df.logbirths.values)
df['expB'] = nbB.mu; df['SMR_B'] = df.deaths_total / df.expB
print('MODEL B (+ access):', nbB.params.round(4).to_dict())
print(f'  pseudo-R2 (deviance) = {1 - nbB.deviance / nbB.null_deviance:.3f}')
print(f'  access LR improvement: dev {nbA.deviance:.0f} -> {nbB.deviance:.0f} '
      f'(LR chi2 {nbA.deviance - nbB.deviance:.1f}, df 3)')

MODEL B (+ access): {'Intercept': -4.2167, 'idhm_renda': -0.9849, 'idhm_educ': -0.3504, 'logpop': 0.0279, 'dist_nicu_km': 0.0004, 'nicu_per1k': 0.0184, 'ubs_per1k': 0.0047}
  pseudo-R2 (deviance) = 0.172
  access LR improvement: dev 6993 -> 6970 (LR chi2 23.1, df 3)


## 5. SMR uncertainty - Byar 95% CI + empirical-Bayes shrinkage

Byar's approximation gives an exact-like Poisson CI on the observed count. Empirical Bayes uses the
NB gamma prior (theta ~ Gamma(r, r), r = 1/alpha) so posterior mean = (O + r) / (E + r); this pulls
thin-denominator municipalities toward their expectation and stops us chasing small-count noise.
**Significant excess = CI lower bound > 1.**

In [7]:
def byar_ci(O, E, z=Z):
    O = np.asarray(O, float); E = np.asarray(E, float)
    lo = O * (1 - 1 / (9 * O) - z / (3 * np.sqrt(O))) ** 3 / E
    hi = (O + 1) * (1 - 1 / (9 * (O + 1)) + z / (3 * np.sqrt(O + 1))) ** 3 / E
    lo = np.where(O == 0, 0.0, lo)
    return lo, hi

df['SMR_A_lo'], df['SMR_A_hi'] = byar_ci(df.deaths_total.values, df.expA.values)
r = 1 / alphaA
df['SMR_A_eb'] = (df.deaths_total + r) / (df.expA + r)
df['sig_excess'] = df.SMR_A_lo > 1
df['sig_deficit'] = df.SMR_A_hi < 1

rel = df[df.deaths_total >= 50].copy()
print(f'--- SMR among {len(rel)} reliable municipalities (>=50 deaths) ---')
print(f'SMR_A: median {rel.SMR_A.median():.2f}, IQR [{rel.SMR_A.quantile(.25):.2f}-{rel.SMR_A.quantile(.75):.2f}]')
print(f'significant EXCESS (CI lower > 1): {int(rel.sig_excess.sum())} munis  '
      f'| significant DEFICIT: {int(rel.sig_deficit.sum())}')
print(f'corr(SMR_A, SMR_B) = {rel.SMR_A.corr(rel.SMR_B):.2f}  (excess is not explained away by access)')
hiA = rel[rel.SMR_A >= rel.SMR_A.quantile(.90)]
still = (hiA.SMR_B >= rel.SMR_B.quantile(.90)).mean()
print(f'top-decile excess in A: {len(hiA)} munis; {still * 100:.0f}% stay top-decile after access adjustment')

--- SMR among 955 reliable municipalities (>=50 deaths) ---
SMR_A: median 1.03, IQR [0.90-1.17]
significant EXCESS (CI lower > 1): 182 munis  | significant DEFICIT: 109
corr(SMR_A, SMR_B) = 0.99  (excess is not explained away by access)
top-decile excess in A: 96 munis; 92% stay top-decile after access adjustment


## 6. Cause layer - where the excess sits (avoidability)

The avoidability claim comes from the **cause**, not the SMR alone. We fit the same structural model
per action group and read the composition of the positive excess among significant-excess
municipalities. If it concentrates in care-amenable causes (prenatal / delivery / newborn) rather
than malformation, the excess is clinically avoidable.

In [8]:
cause_exp = {}
for c in ACTION:
    nbc, _, _ = fit_nb(f'{c} ~ idhm_renda + idhm_educ + logpop', df, df.logbirths.values, outcome=c)
    df[f'exp_{c}'] = nbc.mu
    obs, exp = df[c].sum(), nbc.mu.sum()
    print(f'  {c:12s}: obs {int(obs):>6,} exp {exp:>8,.0f}  national SMR {obs / exp:.2f}')

flag = df[df.sig_excess].copy()
comp = {c: (flag[c] - flag[f'exp_{c}']).clip(lower=0).sum() for c in ACTION}
tot = sum(comp.values())
print(f'\nexcess composition across {len(flag)} significant-excess municipalities '
      f'({int(sum((flag.deaths_total - flag.expA).clip(lower=0))):,} excess deaths):')
for c in ACTION:
    print(f'  {c:12s}: {comp[c]:>7,.0f}  ({comp[c] / tot * 100:4.1f}% of positive excess)')
ranked = sorted(ACTION, key=lambda c: comp[c], reverse=True)
preventable = sum(comp[c] for c in ['prenatal', 'delivery', 'newborn'])
print(f'  -> {ranked[0]} + {ranked[1]} lead; preventable causes are {preventable / tot * 100:.0f}% '
      f'of the excess vs {comp["malformation"] / tot * 100:.0f}% malformation '
      f'=> care-amenable, not irreducible')

# reliable subset (>=50 deaths) -- primary read, free of small-count noise
flag_r = df[df.sig_excess & (df.deaths_total >= 50)].copy()
comp_r = {c: (flag_r[c] - flag_r[f'exp_{c}']).clip(lower=0).sum() for c in ACTION}
tot_r = sum(comp_r.values()); prev_r = sum(comp_r[c] for c in ['prenatal', 'delivery', 'newborn'])
print(f'\n[reliable >=50] excess composition across {len(flag_r)} significant-excess munis '
      f'({int(sum((flag_r.deaths_total - flag_r.expA).clip(lower=0))):,} excess deaths):')
for c in ACTION:
    print(f'  {c:12s}: {comp_r[c]:>7,.0f}  ({comp_r[c] / tot_r * 100:4.1f}%)')
print(f'  -> preventable {prev_r / tot_r * 100:.0f}% vs malformation {comp_r["malformation"] / tot_r * 100:.0f}%')

  prenatal    : obs 73,585 exp   70,927  national SMR 1.04
  delivery    : obs 19,966 exp   19,727  national SMR 1.01
  newborn     : obs 97,583 exp   99,331  national SMR 0.98
  malformation: obs 54,606 exp   54,341  national SMR 1.00

excess composition across 340 significant-excess municipalities (12,962 excess deaths):
  prenatal    :   5,073  (36.2% of positive excess)
  delivery    :   1,395  (10.0% of positive excess)
  newborn     :   5,773  (41.2% of positive excess)
  malformation:   1,776  (12.7% of positive excess)
  -> newborn + prenatal lead; preventable causes are 87% of the excess vs 13% malformation => care-amenable, not irreducible

[reliable >=50] excess composition across 182 significant-excess munis (11,303 excess deaths):
  prenatal    :   4,662  (37.9%)
  delivery    :   1,208  ( 9.8%)
  newborn     :   4,991  (40.5%)
  malformation:   1,453  (11.8%)
  -> preventable 88% vs malformation 12%


## 7. Avoidable burden (one base, a ladder of benchmarks)

The avoidable burden depends entirely on the benchmark municipalities are held to. To keep it
coherent we fix a single base (the 955 reliable municipalities) and a single expectation (the
negative-binomial model), and vary only the target: from performing at the structural mean
(SMR = 1) up to matching the best-performing decile of municipalities of the same structure
(targets read off the SMR distribution). Every row is the same calculation - deaths above
target x expected, summed - only the target changes. We then report how much of the mean-based
excess is statistically significant, and where it sits.

In [9]:
# avoidable-burden ladder: ONE base (reliable municipalities), ONE model expectation (expA),
# only the TARGET performance level changes
rel = df[df.deaths_total >= 50]
base_deaths = rel.deaths_total.sum()

def burden(t):  # deaths above (target x expected), floored at 0, summed over the reliable base
    return (rel.deaths_total - t * rel.expA).clip(lower=0).sum()

targets = [
    ('structural median (SMR p50)', rel.SMR_A.quantile(0.50)),
    ('structural mean (SMR = 1)',   1.0),
    ('best 25% (SMR p25)',          rel.SMR_A.quantile(0.25)),
    ('best quintile (SMR p20)',     rel.SMR_A.quantile(0.20)),
    ('best decile (SMR p10)',       rel.SMR_A.quantile(0.10)),
]
rows = []
for lab, t in targets:
    b = burden(t)
    rows.append({'benchmark': lab, 'target_SMR': round(t, 2),
                 'avoidable_deaths': int(round(b)), 'pct_of_base': round(b / base_deaths * 100, 1)})
ladder = pd.DataFrame(rows)
print('avoidable-burden ladder (base = 955 reliable municipalities, same NB expectation):')
print(ladder.to_string(index=False))
ladder.to_csv(f'{ROOT}/outputs/tables/avoidable_ladder.csv', index=False)

# statistically-confirmed slice of the mean-based excess (significance-gated), and its regional split
df['excess_deaths'] = (df.deaths_total - df.expA).clip(lower=0)
sig_rel = df[df.sig_excess & (df.deaths_total >= 50)]
print(f'\nof which statistically significant (CI lower > 1): {sig_rel.excess_deaths.sum():,.0f} deaths '
      f'in {len(sig_rel)} municipalities (target SMR = 1)')
print('by region (significant excess, reliable):')
for reg, v in sig_rel.groupby('REG').excess_deaths.sum().sort_values(ascending=False).items():
    print(f'  {reg:14s}: {v:>6,.0f}')

avoidable-burden ladder (base = 955 reliable municipalities, same NB expectation):
                  benchmark  target_SMR  avoidable_deaths  pct_of_base
structural median (SMR p50)        1.03             12415          6.4
  structural mean (SMR = 1)        1.00             14724          7.6
         best 25% (SMR p25)        0.90             26757         13.7
    best quintile (SMR p20)        0.87             31128         16.0
      best decile (SMR p10)        0.81             41195         21.2

of which statistically significant (CI lower > 1): 11,303 deaths in 182 municipalities (target SMR = 1)
by region (significant excess, reliable):
  Northeast     :  5,412
  North         :  3,019
  Southeast     :  1,703
  Central-West  :    965
  South         :    205


## 8. Triangulation and data quality

Ill-defined share by SMR tier (does high excess ride on worse cause coding?), and suspect
small-denominator outliers (very high SMR on thin denominators or high ill-def) that we **flag
rather than count** as avoidable without corroboration.

In [10]:
rel = df[df.deaths_total >= 50].copy()
rel['tier'] = pd.cut(rel.SMR_A, [0, 0.8, 1.25, 1.5, np.inf],
                     labels=['deficit(<0.8)', 'normal', 'high(1.25-1.5)', 'very high(>1.5)'])
print(rel.groupby('tier').agg(n=('SMR_A', 'size'),
                              illdef_share=('illdef_share', 'median')).round(3).to_string())
susp = df[(df.SMR_A > 2) & ((df.live_births < 2000) | (df.illdef_share > 0.15))]
print(f'\nsuspect outliers (SMR>2 with thin denominator or high ill-def): {len(susp)} munis '
      f'-> flagged, not counted as avoidable without corroboration')

                   n  illdef_share
tier                              
deficit(<0.8)     76         0.044
normal           728         0.047
high(1.25-1.5)   117         0.049
very high(>1.5)   34         0.064

suspect outliers (SMR>2 with thin denominator or high ill-def): 103 munis -> flagged, not counted as avoidable without corroboration


## 9. Specification stability (robustness)

Does the excess ranking depend on our exact structural model? We refit the expectation with reasonable
enrichments (adding the longevity IDHM sub-index, using the composite IDHM, and adding region fixed
effects) and check whether the same municipalities keep their excess. High rank stability means the
excess is not an artefact of one covariate choice, and that income and education already capture the
relevant structure.

In [11]:
from scipy.stats import spearmanr
rel = df[df.deaths_total >= 50]
top_base = set(rel.index[rel.SMR_A >= rel.SMR_A.quantile(0.90)])
alt = {
    '+ longevity (idhm_long)': 'deaths_total ~ idhm_renda + idhm_educ + logpop + idhm_long',
    'composite IDHM':          'deaths_total ~ idhm + logpop',
    '+ region fixed effects':  'deaths_total ~ idhm_renda + idhm_educ + logpop + C(REG)',
}
print(f'specification stability (reliable municipalities, n={len(rel)}); baseline = Model A')
print(f'{"alternative specification":<28}{"Spearman rho":>13}{"top-decile kept":>17}')
for name, f in alt.items():
    nbs, _, _ = fit_nb(f, df, df.logbirths.values)
    smr_alt = (df.deaths_total / nbs.mu).loc[rel.index]
    rho = spearmanr(rel.SMR_A.values, smr_alt.values)[0]
    top_alt = set(smr_alt.index[smr_alt >= smr_alt.quantile(0.90)])
    kept = len(top_base & top_alt) / len(top_base) * 100
    print(f'{name:<28}{rho:>13.3f}{kept:>14.0f}%')

specification stability (reliable municipalities, n=955); baseline = Model A
alternative specification    Spearman rho  top-decile kept
+ longevity (idhm_long)             0.997            95%
composite IDHM                      0.996            96%
+ region fixed effects              0.970            83%


## 10. Robustness to birth under-registration

Under-registration of live births (SINASC), concentrated in small northern municipalities, would inflate
the mortality rate through the denominator. Two checks. (A) If that drove the excess, every cause SMR
would rise together, so we compare cause-specific SMRs among significant-excess municipalities; a
denominator artefact cannot be cause-specific. (B) We drop the suspect small-denominator / high-ill-def
outliers and recompute the core findings.

In [12]:
# (A) data-artefact test: cause-specific SMR among significant-excess municipalities
sig = df[df.sig_excess & (df.deaths_total >= 50)]
print(f'(A) cause-specific median SMR among {len(sig)} significant-excess municipalities:')
for c in ACTION:
    print(f'    {c:12s}: {(sig[c] / sig[f"exp_{c}"]).median():.2f}')
print('    -> a denominator artefact would raise all four together; malformation lowest = cause-specific')

# (B) exclusion sensitivity: drop suspect small-denominator / high-ill-def outliers, recompute
susp = df[(df.SMR_A > 2) & ((df.live_births < 2000) | (df.illdef_share > 0.15))]
clean = df[~df.index.isin(susp.index)]
relf, relc = df[df.deaths_total >= 50], clean[clean.deaths_total >= 50]
def prev_share(d):
    comp = {c: (d[c] - d[f'exp_{c}']).clip(lower=0).sum() for c in ACTION}
    return sum(comp[c] for c in ['prenatal', 'delivery', 'newborn']) / sum(comp.values()) * 100
def bmean(d, t): return (d.deaths_total - t * d.expA).clip(lower=0).sum()
flagf, flagc = relf[relf.sig_excess], relc[relc.sig_excess]
print(f'\n(B) exclusion sensitivity: {len(susp)} suspect outliers ({int((susp.deaths_total >= 50).sum())} reliable)')
print(f'    significant-excess (reliable): {int(relf.sig_excess.sum())} -> {int(relc.sig_excess.sum())}')
print(f'    preventable share of excess:   {prev_share(flagf):.0f}% -> {prev_share(flagc):.0f}%')
print(f'    avoidable @ SMR=1:             {bmean(relf, 1.0):,.0f} -> {bmean(relc, 1.0):,.0f}')
print(f'    avoidable @ best quintile:     {bmean(relf, relf.SMR_A.quantile(.20)):,.0f} -> {bmean(relc, relc.SMR_A.quantile(.20)):,.0f}')

(A) cause-specific median SMR among 182 significant-excess municipalities:
    prenatal    : 1.29
    delivery    : 1.30
    newborn     : 1.41
    malformation: 1.13
    -> a denominator artefact would raise all four together; malformation lowest = cause-specific

(B) exclusion sensitivity: 103 suspect outliers (3 reliable)
    significant-excess (reliable): 182 -> 179
    preventable share of excess:   88% -> 88%
    avoidable @ SMR=1:             14,724 -> 14,470
    avoidable @ best quintile:     31,128 -> 30,871


## 11. Top significant-excess municipalities + export

In [13]:
print('TOP 10 significant-excess municipalities (>=50 deaths, CI lower > 1), ranked by EB-shrunk SMR:')
top = rel[rel.sig_excess].sort_values('SMR_A_eb', ascending=False).head(10)
for _, x in top.iterrows():
    print(f'  {x.municipio}/{x.uf} ({x.REG[:2]}): obs {int(x.deaths_total)} vs exp {x.expA:.0f}  '
          f'EB-SMR {x.SMR_A_eb:.2f}  (raw {x.SMR_A:.2f} [{x.SMR_A_lo:.2f}-{x.SMR_A_hi:.2f}])')

out_cols = ['CODMUNRES', 'municipio', 'uf', 'REG', 'deaths_total', 'live_births', 'nmr',
            'idhm', 'expA', 'SMR_A', 'SMR_A_lo', 'SMR_A_hi', 'SMR_A_eb',
            'expB', 'SMR_B', 'sig_excess', 'sig_deficit',
            'excess_deaths', 'illdef_share'] + [f'exp_{c}' for c in ACTION]
df[out_cols].to_csv(f'{ROOT}/outputs/tables/excess_mortality.csv', index=False)
print(f'\nwrote outputs/tables/excess_mortality.csv  ({len(df)} rows)')

TOP 10 significant-excess municipalities (>=50 deaths, CI lower > 1), ranked by EB-shrunk SMR:
  Caracaraí/RR (No): obs 156 vs exp 41  EB-SMR 1.92  (raw 3.82 [3.24-4.47])
  Barcelos/AM (No): obs 162 vs exp 59  EB-SMR 1.72  (raw 2.75 [2.34-3.21])
  Itabuna/BA (No): obs 438 vs exp 246  EB-SMR 1.58  (raw 1.78 [1.61-1.95])
  Ilhéus/BA (No): obs 388 vs exp 223  EB-SMR 1.54  (raw 1.74 [1.57-1.92])
  Aracaju/SE (No): obs 1140 vs exp 726  EB-SMR 1.51  (raw 1.57 [1.48-1.66])
  Itaituba/PA (No): obs 393 vs exp 235  EB-SMR 1.49  (raw 1.67 [1.51-1.84])
  Macapá/AP (No): obs 1225 vs exp 803  EB-SMR 1.47  (raw 1.53 [1.44-1.61])
  Eirunepé/AM (No): obs 162 vs exp 89  EB-SMR 1.42  (raw 1.83 [1.56-2.13])
  Parintins/AM (No): obs 323 vs exp 203  EB-SMR 1.41  (raw 1.59 [1.42-1.77])
  Corumbá/MS (Ce): obs 276 vs exp 170  EB-SMR 1.41  (raw 1.62 [1.44-1.82])



wrote outputs/tables/excess_mortality.csv  (5477 rows)


## 12. Tables 1 and 2 (manuscript, exported)

Table 1: negative binomial incidence-rate ratios (exp of coefficients) with 95% CI for Models A and B.
Table 2: the twelve municipalities with the largest significant excess, ranked by empirical-Bayes SMR.

In [14]:
# Table 1: IRR (exp beta) with 95% CI, Models A and B
terms = {'idhm_renda': 'IDHM income sub-index', 'idhm_educ': 'IDHM education sub-index',
         'logpop': 'log(population)', 'dist_nicu_km': 'Distance to nearest NICU (km)',
         'nicu_per1k': 'NICU beds per 1,000 births', 'ubs_per1k': 'Primary-care units per 1,000 births'}
def irr_ci(m, t):
    if t not in m.params.index:
        return 'not in model'
    b, se = m.params[t], m.bse[t]
    return f'{np.exp(b):.2f} ({np.exp(b - 1.96 * se):.2f}-{np.exp(b + 1.96 * se):.2f})'
t1 = pd.DataFrame([{'term': lbl, 'model_A': irr_ci(nbA, k), 'model_B': irr_ci(nbB, k)}
                   for k, lbl in terms.items()])
t1.to_csv(f'{ROOT}/outputs/tables/table1_coefficients.csv', index=False)
print('TABLE 1 (IRR, 95% CI):')
print(t1.to_string(index=False))
print(f'  pseudo-R2: A={1 - nbA.deviance / nbA.null_deviance:.3f}  B={1 - nbB.deviance / nbB.null_deviance:.3f}')

# Table 2: top significant-excess municipalities by EB-SMR, with leading cause group
r2 = df[(df.deaths_total >= 50) & df.sig_excess].copy()
r2['leading_cause'] = r2[ACTION].idxmax(axis=1)
t2 = (r2.sort_values('SMR_A_eb', ascending=False).head(12)
        [['municipio', 'uf', 'REG', 'deaths_total', 'expA', 'SMR_A_eb',
          'SMR_A', 'SMR_A_lo', 'SMR_A_hi', 'leading_cause']])
t2.to_csv(f'{ROOT}/outputs/tables/table2_top_excess.csv', index=False)
print('\nTABLE 2 (top 12 by EB-SMR):')
print(t2.to_string(index=False))

TABLE 1 (IRR, 95% CI):
                               term          model_A          model_B
              IDHM income sub-index 0.38 (0.33-0.44) 0.37 (0.32-0.43)
           IDHM education sub-index 0.63 (0.56-0.72) 0.70 (0.62-0.80)
                    log(population) 1.02 (1.02-1.03) 1.03 (1.02-1.04)
      Distance to nearest NICU (km)     not in model 1.00 (1.00-1.00)
         NICU beds per 1,000 births     not in model 1.02 (1.00-1.04)
Primary-care units per 1,000 births     not in model 1.00 (1.00-1.01)
  pseudo-R2: A=0.162  B=0.172



TABLE 2 (top 12 by EB-SMR):
 municipio uf          REG  deaths_total       expA  SMR_A_eb    SMR_A  SMR_A_lo  SMR_A_hi leading_cause
 Caracaraí RR        North           156  40.831698  1.916026 3.820561  3.244515  4.469377       newborn
  Barcelos AM        North           162  58.854221  1.717543 2.752564  2.344990  3.210600       newborn
   Itabuna BA    Northeast           438 246.457006  1.578066 1.777186  1.614624  1.951680       newborn
    Ilhéus BA    Northeast           388 222.839048  1.536701 1.741167  1.572202  1.923341       newborn
   Aracaju SE    Northeast          1140 726.457548  1.509696 1.569259  1.479475  1.663067      prenatal
  Itaituba PA        North           393 235.418043  1.491963 1.669371  1.508381  1.842863       newborn
    Macapá AP        North          1225 803.182086  1.474979 1.525183  1.440959  1.613046      prenatal
  Eirunepé AM        North           162  88.556977  1.423422 1.829331  1.558460  2.133738       newborn
 Parintins AM        North